In [1]:

from pathlib import Path
import datetime
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio as rio


# reload the module
import importlib

import class_downloadGEE as dlGEE
importlib.reload(dlGEE)


from tqdm import tqdm

In [2]:
ee.Initialize(project='agbd-gedi')
ee.Authenticate()

True

In [93]:
wkdir = Path(r"G:\My Drive\GEE_GEDI_csvs")

out_pat = Path(r"D:\Pesquisa\PhD\IGARS2025\DatasetBiomes\output")

csv_list = list(wkdir.glob("*.csv"))

In [94]:
len(csv_list)

41

In [84]:
for j in tqdm(range(0, len(csv_list))):
    
    out_file = Path(out_pat, f"{csv_list[j].stem}_S1.csv")
    
    if out_file.exists():
        continue

    df = pd.read_csv(csv_list[j], header=0)
    df_columns = ['x', 'y', 'abgbd', 'date']
    # Set columns for the dataframe
    df.columns = df_columns
    df['date'] = df['date'].astype(str).str[:6]  # Ensure only the first 6 characters
    df['date'] = pd.to_datetime(df['date'], format='%Y%m', errors='coerce')

    unique_coords = df.groupby(['x', 'y']).size().reset_index()
    a_results = []
    d_results = []
    
    
    for i in tqdm(range(0, len(unique_coords)), leave=False):  # Iterate over all unique coordinates
        date_range = ("2019-03-05", "2023-04-05")
    

        current_point = ee.Geometry.Point(unique_coords.iloc[i].x , unique_coords.iloc[i].y)


        # Load the Sentinel-1 ImageCollection
        sentinel_1 = (
                    ee.ImageCollection('COPERNICUS/S1_GRD')
                    .filterDate("2019-03-05", "2023-04-05")
                    .filterBounds(current_point)
                    )

        # Filter the Sentinel-1 collection by metadata properties.
        vv_vh_iw = (
            sentinel_1.filter(
                # Filter to get images with VV and VH dual polarization.
                ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')
            )
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .filter(
                # Filter to get images collected in interferometric wide swath mode.
                ee.Filter.eq('instrumentMode', 'IW')
            )
        )

        # Separate ascending and descending orbit images into distinct collections.
        vv_vh_iw_asc = vv_vh_iw.filter(
            ee.Filter.eq('orbitProperties_pass', 'ASCENDING')
        )
        vv_vh_iw_desc = vv_vh_iw.filter(
            ee.Filter.eq('orbitProperties_pass', 'DESCENDING')
        )


        # Function to extract VV and VH values at the point
        def extract_vv_vh(image):
            vv = image.select('VV').reduceRegion(
                reducer=ee.Reducer.mean(), geometry=current_point, scale=10
            ).get('VV')
            vh = image.select('VH').reduceRegion(
                reducer=ee.Reducer.mean(), geometry=current_point, scale=10
            ).get('VH')
            date = image.date().format('YYYY-MM-dd')
            orbit_pass = image.get('orbitProperties_pass')
            return ee.Feature(None, {'VV': vv, 'VH': vh, 'date': date, 'orbit': orbit_pass, 'x': current_point.coordinates().get(0), 'y': current_point.coordinates().get(1)})

        # Map extraction function over ascending and descending collections
        vv_vh_asc_features = vv_vh_iw_asc.map(extract_vv_vh)
        vv_vh_desc_features = vv_vh_iw_desc.map(extract_vv_vh)

        # Convert FeatureCollections to lists for exporting as dictionaries
        asc_data = vv_vh_asc_features.getInfo()
        desc_data = vv_vh_desc_features.getInfo()

        # Extract as Python-readable results
        asc_results = [{'VV': f['properties']['VV'], 'VH': f['properties']['VH'], 'date': f['properties']['date'], 'orbit': f['properties']['orbit'] , 'x': unique_coords.iloc[i].x, 'y': unique_coords.iloc[i].y} for f in asc_data['features']]
        desc_results = [{'VV': f['properties']['VV'], 'VH': f['properties']['VH'], 'date': f['properties']['date'], 'orbit': f['properties']['orbit'], 'x': unique_coords.iloc[i].x, 'y': unique_coords.iloc[i].y} for f in desc_data['features']]


        a_results.append([asc_results])
        d_results.append([desc_results])
        
        df_d = pd.DataFrame(d_results[0][0])
        df_a = pd.DataFrame(a_results[0][0])

        df_out = pd.concat([df_d, df_a], axis=1)
        
        

        df1 = pd.DataFrame(df_out)
        df1.to_csv(out_file, index=False)

  0%|          | 0/41 [00:00<?, ?it/s]